# Managed vs Unmanaged Nodes Dashboard

This notebook:
- Uses RESTful HTTP calls (no Python SDK dependency)
- Reads credentials from environment variables:
  - SYSTEMLINK_API_KEY
  - SYSTEMLINK_HTTP_URI
- Builds month-by-month snapshots
- Outputs: result = [summary, detail, month]
- Uses sb.glue('result', result) for Grafana


In [1]:
import os
import requests
import pandas as pd
import numpy as np
from datetime import datetime
from datetime import timezone
from dateutil.relativedelta import relativedelta
import scrapbook as sb


## Load Environment Variables

In [2]:
api_key = os.getenv("SYSTEMLINK_API_KEY")
sl_uri = os.getenv("SYSTEMLINK_HTTP_URI")

BASE_URL = sl_uri.rstrip("/")

headers = {
    'Content-Type': 'application/json',
    'x-ni-api-key': api_key
}

months_to_build = 12
license_duration = 12

## Query Systems via REST

In [ ]:
systems_url = f"{BASE_URL}/nisysmgmt/v1/query-systems"
all_systems = []
take = 1000
now = datetime.now(timezone.utc).replace(day=1, hour=0, minute=0, second=0, microsecond=0)
eval_duration = now - relativedelta(months=license_duration)

        
skip = 0
while True:
    payload = {
        "filter": 'grains.data.host != null and grains.data.host != "" and connected.data.state != "VIRTUAL" and (activation.data.activated == true or activation.data.activated == null)',
        "skip": skip,
        "take": take,
        "projection": "new(id, grains.data.host as host,  createdTimestamp, connected.lastUpdatedTimestamp as  lastUpdated)"
    }

    response = requests.post(systems_url, json=payload, headers=headers)
    response.raise_for_status()
    
    data = response.json().get("data", [])
    if not data:
        break
    
    all_systems.extend(data)
    skip += take

# Create DataFrame
system_df = pd.DataFrame(all_systems)

if not system_df.empty:
    # 1. Rename 'host' to 'Host Name'
    system_df = system_df.rename(columns={"host": "Host Name"})
    
    # 2. Strip whitespace from 'Host Name' column
    # We use .astype(str) as a safety measure before applying string functions
    system_df["Host Name"] = system_df["Host Name"].astype(str).str.strip()
    
    # 3. Clean up the date format
    system_df['createdTimestamp'] = pd.to_datetime(system_df['createdTimestamp'])
    system_df['lastUpdated'] = pd.to_datetime(system_df['lastUpdated'])
    
    # 4. Sort by createdTimestamp (oldest first) so we keep the earliest record of the system
    system_df = system_df.sort_values("createdTimestamp", ascending=True)

    # 5. Reorder/Select columns
    system_df = system_df[["id", "Host Name", "createdTimestamp", "lastUpdated"]]
else:
    system_df = pd.DataFrame(columns=["id", "Host Name", "createdTimestamp", "lastUpdated"])

# Reset index so the row numbers are clean (0, 1, 2...) after dropping rows
system_df = system_df.reset_index(drop=True)

print(f"Total valid systems found: {len(system_df)}")

Total valid systems found: 305


In [ ]:
## Query Virtual Systems via REST

In [5]:
virtual_systems = []
skip = 0
while True:
    payload = {
        "filter": 'connected.data.state == "VIRTUAL" ',
        "skip": skip,
        "take": take,
        "projection": "new(id, alias as host, createdTimestamp, createdTimestamp as lastUpdated)"
    }

    response = requests.post(systems_url, json=payload, headers=headers)
    response.raise_for_status()
    
    data = response.json().get("data", [])
    if not data:
        break
    
    virtual_systems.extend(data)
    skip += take

# Create DataFrame
virtual_df = pd.DataFrame(virtual_systems)

if not virtual_df.empty:
    # 1. Rename 'host' to 'Host Name'
    virtual_df = virtual_df.rename(columns={"host": "Host Name"})
    
    # 3. Strip
    # We use .astype(str) as a safety measure before applying string functions
    virtual_df["Host Name"] = virtual_df["Host Name"].astype(str).str.strip()
    
    # 4. Clean up the date format
    virtual_df['createdTimestamp'] = pd.to_datetime(virtual_df['createdTimestamp'])
    virtual_df['lastUpdated'] = pd.to_datetime(virtual_df['lastUpdated'])
    
    # 5. Remove Duplicates
    # Sort by createdTimestamp (oldest first) so we keep the earliest record of the system
    virtual_df = virtual_df.sort_values("createdTimestamp", ascending=True)
    #virtual_df = virtual_df.drop_duplicates(subset=["Host Name"], keep="first")
    
    # 6. Reorder/Select columns
    virtual_df = virtual_df[["id", "Host Name", "createdTimestamp", "lastUpdated"]]
else:
    virtual_df = pd.DataFrame(columns=["id", "Host Name", "createdTimestamp", "lastUpdated"])

# Reset index so the row numbers are clean (0, 1, 2...) after dropping rows
virtual_df = virtual_df.reset_index(drop=True)

print(f"Total virtual systems found: {len(virtual_df)}")

Total virtual systems found: 114


In [ ]:
## Query Stale Systems via REST

In [ ]:
stale_systems = []
skip = 0
while True:
    payload = {
        "filter": f'''grains.data.host != null and grains.data.host != "" and connected.data.state != "VIRTUAL"  and (connected.lastUpdatedTimestamp < "{eval_duration.isoformat()}" and connected.data.state != "CONNECTED") and (activation.data.activated == true or activation.data.activated == null)''',
        "skip": skip,
        "take": take,
        "projection": "new(id, grains.data.host as host, createdTimestamp, connected.lastUpdatedTimestamp as lastUpdated)"
    }

    response = requests.post(systems_url, json=payload, headers=headers)
    response.raise_for_status()
    
    data = response.json().get("data", [])
    if not data:
        break
    
    stale_systems.extend(data)
    skip += take
    
# Create DataFrame
stale_df = pd.DataFrame(stale_systems)

if not stale_df.empty:
    # 1. Rename 'host' to 'Host Name'
    stale_df = stale_df.rename(columns={"host": "Host Name"})
    
    # 3. Strip
    # We use .astype(str) as a safety measure before applying string functions
    stale_df["Host Name"] = stale_df["Host Name"].astype(str).str.strip()
    
    # 4. Clean up the date format
    stale_df['createdTimestamp'] = pd.to_datetime(stale_df['createdTimestamp'])
    stale_df['lastUpdated'] = pd.to_datetime(stale_df['lastUpdated'])
    
    # 5. Remove Duplicates
    # Sort by createdTimestamp (oldest first) so we keep the earliest record of the system
    stale_df = stale_df.sort_values("createdTimestamp", ascending=True)
    #stale_df = stale_df.drop_duplicates(subset=["Host Name"], keep="first")
    
    # 6. Reorder/Select columns
    stale_df = stale_df[["id", "Host Name", "createdTimestamp", "lastUpdated"]]
else:
    stale_df = pd.DataFrame(columns=["id", "Host Name", "createdTimestamp", "lastUpdated"])

print(f"Total stale systems found: {len(stale_df)}")

Total stale systems found: 128


## Freeze Snapshot Time

In [8]:
#now = datetime.now(timezone.utc).replace(day=1, hour=0, minute=0, second=0, microsecond=0)
current_month = now.strftime("%Y-%m")

## Build Monthly Snapshots

In [9]:
df_summary_list = []
detail_list = []

for m in range(months_to_build):
    snapshot_date = now - relativedelta(months=m)
    month_label = snapshot_date.strftime("%Y-%m")

    # --- Managed Snapshot ---
    managed_snapshot = system_df[
        system_df["createdTimestamp"] <= snapshot_date
    ].copy()
    managed_snapshot["Node Type"] = "Managed"

    # Assign Status to Managed    
    stale_cutoff = snapshot_date - relativedelta(months=license_duration)
    
    stale_ids = set(
        system_df[
            pd.to_datetime(system_df["lastUpdated"], errors="coerce") <= stale_cutoff
        ]["id"]
    )

    managed_snapshot["Status"] = np.where(
        managed_snapshot["id"].isin(stale_ids),
        "Inactive",
        "Active"
    )

    # --- Unmanaged Definition (Test Monitor) ---
    window_start = snapshot_date - relativedelta(months=license_duration)
    results_filter = (
        f'updatedAt >= "{window_start.isoformat()}" '
        f'and updatedAt <= "{snapshot_date.isoformat()}"'
    )

    results_url = f"{BASE_URL}/nitestmonitor/v2/query-result-values"
    results_payload = {"field": "HOST_NAME", "filter": results_filter}

    results_response = requests.post(results_url, json=results_payload, headers=headers)
    results_response.raise_for_status()

    df_results = pd.DataFrame(results_response.json(), columns=["Host Name"])
    df_results["Host Name"] = df_results["Host Name"].astype(str).str.strip()
    df_results = df_results.drop_duplicates()

    # Normalize casing
    df_results["Host Name"] = df_results["Host Name"].fillna("").str.upper()
    virtual_df["Host Name"] = virtual_df["Host Name"].fillna("").str.upper()
    managed_hosts = managed_snapshot["Host Name"].fillna("").str.upper()
    
    # --- Separate sources ---
    # Results (no id)
    df_results["id"] = None
    df_results["createdTimestamp"] = pd.NaT
    df_results["lastUpdated"] = pd.NaT
    
    # Virtual already has id + timestamps
    
    # --- Combine WITHOUT collapsing by host ---
    combined = pd.concat([
        df_results[["id", "Host Name", "createdTimestamp", "lastUpdated"]],
        virtual_df[["id", "Host Name", "createdTimestamp", "lastUpdated"]]
    ], ignore_index=True)
    
    # --- Remove managed hosts ---
    unmanaged_snapshot = combined[
        ~combined["Host Name"].isin(managed_hosts)
    ].copy()
    
    unmanaged_snapshot["Node Type"] = "Unmanaged"


    # Assign Status to Unmanaged
    virtual_hosts = set(virtual_df["Host Name"])
    
    unmanaged_snapshot["Status"] = np.where(
        unmanaged_snapshot["Host Name"].isin(virtual_hosts),
        "Virtual",
        "Active"
    )


    # --- Summary ---
    summary_df = pd.DataFrame({
        "Node Type": ["Managed", "Unmanaged"],
        "Count": [len(managed_snapshot), len(unmanaged_snapshot)],
        "month": month_label
    })

    df_summary_list.append(summary_df)

    # --- Detail Table (FULL METADATA) ---
    cols = ["id", "Host Name", "Node Type", "Status", "createdTimestamp", "lastUpdated"]
    
    # Managed
    managed_detail = managed_snapshot.copy()
    managed_detail["Node Type"] = "Managed"
    
    # Unmanaged
    unmanaged_detail = unmanaged_snapshot.copy()
    unmanaged_detail["Node Type"] = "Unmanaged"
    
    # Ensure datetime consistency
    for df in [managed_detail, unmanaged_detail]:
        df["createdTimestamp"] = pd.to_datetime(df["createdTimestamp"], errors="coerce")
        df["lastUpdated"] = pd.to_datetime(df["lastUpdated"], errors="coerce")
    
    # Select final columns
    managed_detail = managed_detail[cols]
    unmanaged_detail = unmanaged_detail[cols]
    
    # Combine
    detail_df = pd.concat([managed_detail, unmanaged_detail], ignore_index=True)

    
    detail_df["month"] = month_label
    detail_list.append(detail_df)



## Final Grafana Output

In [10]:
final_df_summary = pd.concat(df_summary_list, ignore_index=True)
final_detail = pd.concat(detail_list, ignore_index=True)

final_df_summary = final_df_summary.sort_values("month")

detail_current = final_detail[
    final_detail["month"] == current_month
]

summary_current = pd.DataFrame({
    "Node Type": [
        "Managed",
        "Unmanaged",
        "Stale",
        "Virtual"
    ],
    "Count": [
        len(detail_current[detail_current["Node Type"] == "Managed"]),
        len(detail_current[detail_current["Node Type"] == "Unmanaged"]),
        len(detail_current[detail_current["Status"] == "Inactive"]),
        len(detail_current[detail_current["Status"] == "Virtual"])
    ]
})

df_summary_current = summary_current

# ------------------------------------------------------------------
# Make JSON-safe copies for scrapbook output
# ------------------------------------------------------------------
final_detail_out = final_detail.copy()
final_df_summary_out = final_df_summary.copy()
df_summary_current_out = df_summary_current.copy()

# Convert datetime columns to ISO strings, preserving missing values as None
for df in [final_detail_out, final_df_summary_out, df_summary_current_out]:
    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            df[col] = df[col].apply(lambda x: x.isoformat() if pd.notnull(x) else None)

# Replace remaining NaN/NaT with None so JSON serialization succeeds
final_detail_out = final_detail_out.where(pd.notnull(final_detail_out), None)
final_df_summary_out = final_df_summary_out.where(pd.notnull(final_df_summary_out), None)
df_summary_current_out = df_summary_current_out.where(pd.notnull(df_summary_current_out), None)

# ------------------------------------------------------------------
# Build scrapbook payloads
# ------------------------------------------------------------------
df_dict = {
    'columns': pd.io.json.build_table_schema(df_summary_current_out, index=False)['fields'],
    'values': df_summary_current_out.values.tolist(),
}

summary = {
    'type': 'data_frame',
    'id': 'df_summary_current',
    'data': df_dict,
}

df_dict = {
    'columns': pd.io.json.build_table_schema(final_detail_out, index=False)['fields'],
    'values': final_detail_out.values.tolist(),
}

detail = {
    'type': 'data_frame',
    'id': 'final_detail',
    'data': df_dict,
}

df_dict = {
    'columns': pd.io.json.build_table_schema(final_df_summary_out, index=False)['fields'],
    'values': final_df_summary_out.values.tolist(),
}

month = {
    'type': 'data_frame',
    'id': 'final_df_summary',
    'data': df_dict,
}

result = [summary, detail, month]

sb.glue("result", result)


In [11]:
summary

{'type': 'data_frame',
 'id': 'df_summary_current',
 'data': {'columns': [{'name': 'Node Type', 'type': 'string'},
   {'name': 'Count', 'type': 'integer'}],
  'values': [['Managed', 297],
   ['Unmanaged', 153],
   ['Stale', 128],
   ['Virtual', 114]]}}